In [15]:
from pathlib import Path

import ray
import torch

from climanet.tune import run_tune
from climanet.dataset import DataLoaderConfig, STDataset
from climanet.predict import predict_monthly_var, PredictionConfig
from climanet.utils import set_seed, configure_compute_resources, data_preparation, read_st_data

import xarray as xr

In [2]:
data_folder = Path("./eso4clima/dc_data")
run_dir = Path("./tune_daily").resolve()

var_name = "tos"

# 1 month train, 1 month validation and test
daily_data = xr.open_mfdataset(data_folder / f"202101_day_ERA5dc_masked_{var_name}.nc")
daily_data_validation = xr.open_mfdataset(data_folder / f"202102_day_ERA5dc_masked_{var_name}.nc")
daily_data_test = xr.open_mfdataset(data_folder / f"202103_day_ERA5dc_masked_{var_name}.nc")

monthly_data = xr.open_mfdataset(data_folder / f"202101_mon_ERA5dc_full_{var_name}.nc")
monthly_data_validation = xr.open_mfdataset(data_folder / f"202102_mon_ERA5dc_full_{var_name}.nc")
monthly_data_test = xr.open_mfdataset(data_folder / f"202103_mon_ERA5dc_full_{var_name}.nc")

file_name = data_folder / "era5_lsm_bool.nc"  # downloded from era5 and regridded using the function `regrid_to_boundary_centered_grid`
lsm_mask = xr.open_dataset(file_name)

### prepare data for tuning

In [3]:
# coordinates of subset
lon_subset = slice(-50, 50)  # one lon -179.9 is nan, check data
lat_subset = slice(-30, 10)

daily_subset = daily_data.sel(lon=lon_subset, lat=lat_subset)
monthly_subset = monthly_data.sel(lon=lon_subset, lat=lat_subset)
lsm_subset = lsm_mask.sel(lon=lon_subset, lat=lat_subset)  # True=Land

daily_validation_subset = daily_data_validation.sel(lon=lon_subset, lat=lat_subset)
monthly_validation_subset = monthly_data_validation.sel(lon=lon_subset, lat=lat_subset)

daily_test_subset = daily_data_test.sel(lon=lon_subset, lat=lat_subset)
monthly_test_subset = monthly_data_test.sel(lon=lon_subset, lat=lat_subset)

print(daily_subset[var_name].shape, monthly_subset[var_name].shape)  # (time, lat, lon)

(31, 160, 400) (1, 160, 400)


In [4]:
data_dir = Path(f"{run_dir}/data_train")
input_da, input_da_nan_mask, monthly_da, padded_days_mask, time_features = data_preparation(
    daily_subset[var_name], monthly_subset[var_name], calculate_residuals=True, is_hourly=False, save_to_zarr=True, run_dir=data_dir,
)

In [5]:
data_dir = f"{run_dir}/data_validation"
input_da, input_da_nan_mask, monthly_da, padded_days_mask, time_features = data_preparation(
    daily_validation_subset[var_name], monthly_validation_subset[var_name], calculate_residuals=True, is_hourly=False, save_to_zarr=True, run_dir=data_dir,
)

### config for hyper parameter tuning

In [ ]:
data_config_train = {
    "input_data_dir": f"{run_dir}/data_train",
    "land_mask_data": ray.put(lsm_subset["lsm"]),
    "load_lazy": False,  # one year fits in memory
    "patch_size": (1, 40, 40),
    "stride": (20, 20),
    "var_name": var_name,
}

data_config_validation = {
    "input_data_dir": f"{run_dir}/data_validation",
    "land_mask_data": ray.put(lsm_subset["lsm"]),
    "load_lazy": False,  # one year fits in memory
    "patch_size": (1, 40, 40),
    "stride": (20, 20),
    "var_name": var_name,
}

# dont use ray.put() (i.e. object store) when data is large
static_args = {
    "data_config_train": data_config_train,
    "data_config_validation": data_config_validation,
    "is_hourly": True,
    "var_name": var_name,
    "max_num_epochs": 1,
    "num_trials": 2,  # this is num_samples in ray.tune.TuneConfig
    "cpu_per_trial": 1,
    "gpu_per_trial": 0,
    "run_dir": run_dir,
    "device": "cpu",
    "dataloader_num_workers": 1,
    "dataloader_persistent_workers": True,
    "dataloader_multiprocessing_context": None,  # load_lazy is False
    "num_epoch": 1,
    "max_concurrent_trials": 1,  # less than GPUs per node (4) avoid OOM
    "experiment_name": "sst_01",  # not a long name
}

# parameters to tune
tune_config = {
    "patch_size": ray.tune.grid_search([2, 4]),
    "overlap": 2,
    "embed_dim": 32,
    "dropout": 0.0,
    "hidden": 32,
    "spatial_depth": 3,
    "spatial_heads": 4,
    "optimizer_lr": 1e-1,
    "batch_config": {"batch_size": 10, "accumulation_steps": 5},
}

### Run ray tune

In [7]:
results = run_tune(tune_config, static_args)

ray.shutdown()

2026-08-07 15:16:27,661	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/sarah/GitHub/ClimaNet/notebooks/tune_daily/sst_01' in 0.0069s.
2026-08-07 15:16:27,668	INFO tune.py:1039 -- Total run time: 71.26 seconds (71.20 seconds for the tuning loop).


### Inspect the output

In [8]:
results.get_best_result("loss", "min")

Result(
  metrics={'loss': 0.15448503834860666},
  path='/home/sarah/GitHub/ClimaNet/notebooks/tune_daily/sst_01/_train_0804a_00001_1_patch_size=4_2026-08-07_15-15-37',
  filesystem='local',
  checkpoint=Checkpoint(filesystem=local, path=/home/sarah/GitHub/ClimaNet/notebooks/tune_daily/sst_01/_train_0804a_00001_1_patch_size=4_2026-08-07_15-15-37/checkpoint_000000)
)

### Check best model

In [9]:
# find the path to best model
if not ray.is_initialized():
    ray.init()

experiment_path = results.experiment_path  # or add the path above manually as Path("./runs_daily/_train_2026-07-23_10-11-56\").resolve()"
analysis = ray.tune.ExperimentAnalysis(experiment_path)
best_result = analysis.get_best_trial("loss", "min")
best_checkpoint = best_result.checkpoint
model_path = Path(best_checkpoint.path) / "checkpoint.pt"

2026-08-07 15:17:20,411	INFO worker.py:2024 -- Started a local Ray instance.


In [10]:
data_dir = f"{run_dir}/data_test"
input_da, input_da_nan_mask, monthly_da, padded_days_mask, time_features = data_preparation(
    daily_test_subset[var_name], monthly_test_subset[var_name], calculate_residuals=True, is_hourly=False, save_to_zarr=True, run_dir=data_dir,
)

In [11]:
# read data 
data_dir = f"{run_dir}/data_test"
input_da, input_da_nan_mask, monthly_da, padded_days_mask, time_features = read_st_data(data_path=data_dir, var_name=var_name)

In [16]:
num_patches = (10, 10)
patch_size = (1, 4, 4)
spatial_patch_size = (patch_size[1]*num_patches[0], patch_size[2]*num_patches[1])
stride = (spatial_patch_size[0] // 5, spatial_patch_size[1] // 5)

dataset_test = STDataset(
    input_da=input_da,
    input_da_nan_mask=input_da_nan_mask,
    monthly_da=monthly_da,
    padded_days_mask=padded_days_mask,
    time_features=time_features,
    land_mask=lsm_subset["lsm"],
    patch_size=(1, *spatial_patch_size),  # based on the patch_size in model
    stride=stride,
    sh_embed_dim=96,
    sh_order_L = 10,
    verbose=True,
    load_lazy=False,
)
print(len(dataset_test))

Creating dataset:
Patch grid (m x i x j): 1 x 16 x 46 = 736 patches
Overlap: 32 pixels (height), 32 pixels (width)
736


In [19]:
# create dataloader config
dataloader_config = DataLoaderConfig(
    batch_size=10,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
    device="cpu",
    multiprocessing_context=None, # keep this when num_workers >0 and lazy data
)
dataloader_config

DataLoaderConfig(batch_size=10, shuffle=True, num_workers=0, pin_memory=False, persistent_workers=False, device='cpu', multiprocessing_context=None)

In [20]:
# create prediction config
prediction_config = PredictionConfig(
    calculate_residuals=True,
    return_numpy=True,
    save_predictions=False,
    return_loss=True,
    device="cpu",
    verbose=False,
)

In [21]:
test_loss = predict_monthly_var(
    model= model_path,
    dataset=dataset_test,  
    dataloader_config=dataloader_config,
    prediction_config=prediction_config,
    run_dir=run_dir,
)
test_loss

0.15355809959205421